### Supabase DB 및 Gemini 임베딩을 이용한 RAG 데이터 적재 (v1.2.3)
- 대상 CSV: main_v1_1_9.csv
- 주요 업데이트 내역:
  1. `target_group` 대신 `target_tags`를 임베딩의 대상 키워드로 사용
  2. `target_tags` DB 저장 시 리스트(ARRAY) 형태로 후처리
  3. DB 내 최근 14일 데이터를 조회하여 신규 수집된(DB에 없는) 데이터만 필터링하여 임베딩 및 적재
  4. 노트북 재시작 시에도 가장 최근 임베딩된 JSON을 불러와 DB 적재를 이어서 할 수 있도록 개선
  5. 임베딩 텍스트에 `support_type(지원내용)` 추가 및 빈 값(Null) 항목 라인 자동 제거 로직 적용
- 필요한 키 : GOOGLE_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY

In [3]:
import os
import glob
import google.generativeai as genai
import pandas as pd
import json
import time
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv(override=True)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")

# 클라이언트 초기화
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

print("환경 설정 및 Supabase 클라이언트 초기화 완료")

환경 설정 및 Supabase 클라이언트 초기화 완료


### 1. DB 기적재 데이터 검증 및 신규 데이터 필터링
- API 수집 주기가 7일이므로, 안전하게 14일 이전까지의 DB 데이터를 불러와 신규 수집된 데이터와 비교합니다.
- DB에 이미 적재된 `(source, source_id)` 세트를 걸러내고, 새롭게 추가된 데이터만 도출합니다.

In [4]:
# 1. DB에서 최근 14일 이내 생성된 (source, source_id) 조합 가져오기
fourteen_days_ago = (datetime.now() - timedelta(days=14)).isoformat()
res = supabase.table('announcements').select('source, source_id').gte('created_dt', fourteen_days_ago).execute()
db_records = res.data

# DB에 존재하는 (source, source_id) 세트 생성
existing_keys = set((row['source'], str(row['source_id'])) for row in db_records)
print(f"최근 14일 내 DB에 기적재된 데이터 건수: {len(existing_keys)}건")

# 2. CSV 파일 로드 (가장 최신 main 데이터)
csv_path = "data/csv/main/main_v1_1_11.csv"
df = pd.read_csv(csv_path)
print(f"CSV 전체 데이터 건수: {len(df)}건")

# 3. DB에 없는 신규 데이터만 필터링
new_df = df[~df.apply(lambda row: (row['source'], str(row['source_id'])) in existing_keys, axis=1)].copy()
print(f"\n🚀 DB에 적재되지 않은 신규 데이터 건수(임베딩 및 적재 대상): {len(new_df)}건")

최근 14일 내 DB에 기적재된 데이터 건수: 445건
CSV 전체 데이터 건수: 565건

🚀 DB에 적재되지 않은 신규 데이터 건수(임베딩 및 적재 대상): 120건


### 2. 데이터 전처리 및 임베딩 텍스트 구성
- 기존 `target_group`에 있던 긴 문자열 등의 노이즈를 제거하기 위해, 임베딩 '대상' 컬럼 값을 `target_tags`로 교체합니다.
- **[New]** 임베딩 텍스트에 `support_type(지원내용)`을 추가하여 사용자의 혜택 중심 검색(예: 지원금, 공간지원 등) 성능을 올립니다.
- **[New]** 값이 비어있는 항목은 억지로 '제한없음'을 넣지 않고, 아예 해당 줄(Line) 자체를 삭제하여 깔끔한 임베딩 텍스트를 만듭니다.

In [5]:
# 임베딩용 텍스트 구성
def combine_features(row):
    def safe_str(val):
        return str(val).strip() if pd.notna(val) else ''
    
    parts = [
        f"제목: {safe_str(row.get('title'))}",
        f"카테고리: {safe_str(row.get('s_category'))}",
        f"지역: {safe_str(row.get('region'))}",
        f"대상: {safe_str(row.get('target_tags'))}",
        f"지원내용: {safe_str(row.get('support_type'))}",
        f"요약: {safe_str(row.get('summary'))}",
    ]
    
    # 값이 비어있는 항목은 아예 줄을 생성하지 않고 완전히 삭제
    return "\n".join(p for p in parts if p.split(": ", 1)[1].strip())

if len(new_df) > 0:
    new_df['combined_text'] = new_df.apply(combine_features, axis=1)
    display(new_df[['title', 'combined_text']].head(2))

,title,combined_text
445,부산 청끌기업 발굴·매칭 지원사업(청끌기업),제목: 부산 청끌기업 발굴·매칭 지원사업(청끌기업)\n카테고리: 인력/일자리\n지역...
446,부산 청년고용 우수기업 인증·지원,제목: 부산 청년고용 우수기업 인증·지원\n카테고리: 인력/일자리\n지역: 부산광역...


### 3. Gemini 임베딩 생성 및 JSON 저장
- 시분초를 제외하고 **날짜(YYYYMMDD)** 기준으로 파일명 생성
- 임베딩이 완료되면 메모리가 아닌 **물리적 JSON 파일로 저장**하여 파이프라인 분리

In [6]:
# 임베딩 생성 함수
def get_embedding(text):
    model_name = "models/gemini-embedding-001"
    result = genai.embed_content(
        model=model_name,
        content=text,
        task_type="retrieval_document",
        output_dimensionality=768
    )
    return result['embedding']

def split_text(text, max_length=1500):
    if len(text) <= max_length:
        return [text]
    return [text[i:i + max_length] for i in range(0, len(text), max_length)]

# 파일명: 시분초 제외, 날짜(YYYYMMDD)만 포함
script_version = "v1_2_3"
today_str = datetime.now().strftime("%Y%m%d") 
embedding_dir = "data/embedding"
embedding_file = os.path.join(embedding_dir, f"embedded_announcements_{script_version}_{today_str}.json")
os.makedirs(embedding_dir, exist_ok=True)

insert_data = []

if len(new_df) > 0:
    print(f"새로운 임베딩을 생성합니다. (API 호출 발생, 대상: {len(new_df)}건)")
    
    for idx, row in new_df.iterrows():
        full_text = row['combined_text']
        chunks = split_text(full_text, max_length=1500)
        
        for chunk in chunks:
            try:
                embedding = get_embedding(chunk)
                data = row.to_dict()
                
                for key, val in data.items():
                    if val == '확인필요' or pd.isna(val):
                        data[key] = None
                
                data["content"] = chunk
                data["embedding"] = embedding
                
                if 'combined_text' in data:
                    del data['combined_text']
                if 'additional_conditions' in data:
                    del data['additional_conditions']
                
                insert_data.append(data)
                time.sleep(0.5)
            except Exception as e:
                print(f"임베딩 생성 오류 (Index {idx}): {e}")
                
    # JSON 파일로만 저장 완료
    with open(embedding_file, 'w', encoding='utf-8') as f:
        json.dump(insert_data, f, ensure_ascii=False, indent=2)
    print(f"✅ 신규 임베딩 데이터가 JSON 파일로 저장되었습니다: {embedding_file}")
else:
    print("신규 임베딩 대상 데이터가 없습니다.")

새로운 임베딩을 생성합니다. (API 호출 발생, 대상: 120건)
✅ 신규 임베딩 데이터가 JSON 파일로 저장되었습니다: data/embedding\embedded_announcements_v1_2_3_20260507.json


### 4. JSON 파일 로드 및 DB 적재 (독립 실행 가능)
- 노트북을 껐다가 다시 켜더라도, **가장 최근에 저장된 JSON 파일을 자동으로 찾아 로드**합니다.
- 로드된 데이터를 DB 스키마에 맞게 매핑/후처리하고 Supabase에 Insert 합니다.

In [7]:
# -----------------------------------------------------
# Step A: 저장된 최신 JSON 임베딩 파일 찾아서 읽기
# -----------------------------------------------------
embedding_dir = "data/embedding"
script_version = "v1_2_3"
search_pattern = os.path.join(embedding_dir, f"embedded_announcements_{script_version}_*.json")

file_list = glob.glob(search_pattern)
loaded_data = []

if not file_list:
    print("⚠️ 불러올 임베딩 파일이 없습니다.")
else:
    # 파일 수정 시간 기준으로 가장 최근 파일 선택
    latest_file = max(file_list, key=os.path.getmtime)
    print(f"가장 최근 임베딩 파일을 불러옵니다: {latest_file}")
    
    with open(latest_file, 'r', encoding='utf-8') as f:
        loaded_data = json.load(f)
    print(f"✅ {len(loaded_data)}건의 데이터를 로드했습니다.")

# -----------------------------------------------------
# Step B: DB 컬럼 매핑 및 후처리 로직
# -----------------------------------------------------
if len(loaded_data) > 0:
    # 1. DB 스키마 조회
    headers = {
        "apikey": SUPABASE_SERVICE_KEY,
        "Authorization": f"Bearer {SUPABASE_SERVICE_KEY}"
    }
    res = requests.get(f"{SUPABASE_URL}/rest/v1/", headers=headers)
    spec = res.json()
    table_columns = set(spec['definitions']['announcements']['properties'].keys())
    
    # 2. 매핑 및 형변환 로직
    rename_map = {
        'apply_start': 'apply_start_dt',
        'apply_end': 'apply_end_dt'
    }
    
    def parse_date(date_val):
        if not date_val: return None
        date_str = str(date_val).strip()
        if len(date_str) == 8 and date_str.isdigit():
            return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
        elif len(date_str) >= 10 and "-" in date_str:
            return date_str
        return None

    filtered_data = []
    
    for data in loaded_data:
        filtered_row = {}
        for k, v in data.items():
            mapped_key = rename_map.get(k, k)
            if mapped_key in table_columns:
                if mapped_key in ['target_age_min', 'target_age_max']:
                    if v is not None:
                        try:
                            filtered_row[mapped_key] = int(float(v))
                        except (ValueError, TypeError):
                            filtered_row[mapped_key] = None
                    else:
                        filtered_row[mapped_key] = None
                elif mapped_key in ['apply_start_dt', 'apply_end_dt']:
                    filtered_row[mapped_key] = parse_date(v)
                elif mapped_key == 'target_tags':
                    if v is not None and str(v).strip() != '':
                        filtered_row[mapped_key] = [tag.strip() for tag in str(v).split(',')]
                    else:
                        filtered_row[mapped_key] = []
                else:
                    filtered_row[mapped_key] = v
        filtered_data.append(filtered_row)
    
    print("✅ 데이터 후처리 및 컬럼 매핑 완료!")
    
    # -----------------------------------------------------
    # Step C: Supabase DB 적재 (Insert)
    # -----------------------------------------------------
    print(f"\n총 {len(filtered_data)}개의 데이터 적재를 시작합니다...")
    batch_size = 1
    
    for i in range(0, len(filtered_data), batch_size):
        batch = filtered_data[i:i + batch_size]
        success = False
        
        for retry in range(5):
            try:
                supabase.table("announcements").insert(batch).execute()
                if (i + 1) % 10 == 0 or (i + 1) == len(filtered_data):
                    print(f"[{i+1}/{len(filtered_data)}] 적재 중...")
                success = True
                break 
            except Exception as e:
                print(f"⚠️ [{i+1}번 데이터] 오류: {e}")
                supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
                time.sleep((retry + 1) * 2)
        
        if not success:
            print(f"❌ {i+1}번 데이터 적재 실패.")
    
    print("\n🎉 DB 적재 완료되었습니다!")

가장 최근 임베딩 파일을 불러옵니다: data/embedding\embedded_announcements_v1_2_3_20260507.json
✅ 120건의 데이터를 로드했습니다.
✅ 데이터 후처리 및 컬럼 매핑 완료!

총 120개의 데이터 적재를 시작합니다...
[10/120] 적재 중...
[20/120] 적재 중...
[30/120] 적재 중...
[40/120] 적재 중...
[50/120] 적재 중...
[60/120] 적재 중...
[70/120] 적재 중...
[80/120] 적재 중...
[90/120] 적재 중...
[100/120] 적재 중...
[110/120] 적재 중...
[120/120] 적재 중...

🎉 DB 적재 완료되었습니다!


### 5. csv와 DB의 지역정보 매핑 확인
- main_vn_n_n.csv와 Supabase announcements.region 값을 분포로 비교

In [10]:
import pandas as pd


# 1. CSV 지역 분포 계산
csv_path = "data/csv/main/main_v1_1_11.csv"
csv_counts = pd.read_csv(csv_path)['region'].value_counts().rename_axis('region').reset_index(name='csv_count')

# 2. Supabase DB 지역 분포 조회 (RPC 대신 select-count 조합 사용)
res = supabase.table('announcements').select('region').execute()
db_df = pd.DataFrame(res.data)
db_counts = db_df['region'].value_counts().rename_axis('region').reset_index(name='db_count')

# 3. 두 결과 병합 (Outer Join)
comparison = pd.merge(csv_counts, db_counts, on='region', how='outer').fillna(0)
comparison['diff'] = comparison['csv_count'] - comparison['db_count']

print("=== CSV vs Supabase 지역 분포 비교 ===")
display(comparison.sort_values('csv_count', ascending=False))

# 차이가 있는 항목만 출력
diff_only = comparison[comparison['diff'] != 0]
if not diff_only.empty:
    print("\n⚠️ 주의: 데이터 개수 차이가 발견되었습니다!")
    display(diff_only)
else:
    print("\n✅ 모든 지역의 데이터 개수가 완벽하게 일치합니다.")


=== CSV vs Supabase 지역 분포 비교 ===


,region,csv_count,db_count,diff
12,전국,173,173,0
15,제주특별자치도,84,84,0
8,서울특별시,45,45,0
1,경기도,38,38,0
4,광주광역시,36,36,0
7,부산광역시,34,34,0
3,경상북도,32,32,0
14,전북특별자치도,21,21,0
0,강원특별자치도,18,18,0
2,경상남도,15,15,0



✅ 모든 지역의 데이터 개수가 완벽하게 일치합니다.
